# Classification Wrap-Up

PKS가 정리해 둔 `mnist`, `cifar10`, `cifar100` 결과를 바탕으로 single baseline과 SSML pair 구성을 같은 표 체계로 비교합니다.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "notebook").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_ROOT = REPO_ROOT / "notebook" / "FINAL_WRAPUP"
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from _shared.io import load_curve_file, load_epoch_metrics, load_pks_results, load_run_tree
from _shared.plotting import (
    DATASET_ORDER,
    METHOD_COLORS,
    METHOD_ORDER,
    apply_report_style,
    pretty_dataset,
    pretty_method,
    pretty_model,
    pretty_pair,
    save_figure,
    save_table,
)

apply_report_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

NOTEBOOK_ROOT


## Load


In [ ]:
PKS_ROOT = REPO_ROOT / "notebook" / "results_from_pks" / "result_zip"
classification = load_pks_results(PKS_ROOT)
classification = classification[classification["task"] == "classification"].copy()
classification_file_count = classification["source_path"].nunique()
if classification_file_count != 15:
    raise AssertionError(f"Expected 15 classification PKS CSV files, found {classification_file_count}")

display(classification.head())
classification[["dataset", "source_path"]].drop_duplicates().sort_values(["dataset", "source_path"])


## Normalize


In [ ]:
def stack_curves(curves):
    prepared = []
    for curve in curves:
        if curve is None:
            continue
        arr = np.asarray(curve, dtype=float).reshape(-1)
        if arr.size:
            prepared.append(arr)
    if not prepared:
        raise ValueError("No non-empty curves were provided.")
    min_len = min(len(arr) for arr in prepared)
    return np.vstack([arr[:min_len] for arr in prepared])

def mean_and_std(curves):
    stacked = stack_curves(curves)
    return stacked.mean(axis=0), stacked.std(axis=0)


classification["dataset_label"] = classification["dataset"].map(pretty_dataset)
classification["model_label"] = classification["model"].map(pretty_model)
classification["peer_label"] = classification["peer_model"].map(pretty_model)
classification["metric_percent"] = classification["metric_value"] * 100.0

def config_label(row):
    if row["mode"] == "single_baseline":
        return f"{pretty_model(row['model'])} (single)"
    return f"{pretty_model(row['model'])} | peer={pretty_model(row['peer_model'])}"

classification["config_label"] = classification.apply(config_label, axis=1)

classification_summary = (
    classification.groupby(["dataset", "dataset_label", "config_label"], dropna=False)
    .agg(
        mean_accuracy=("metric_percent", "mean"),
        std_accuracy=("metric_percent", "std"),
        n_seeds=("seed", "nunique"),
    )
    .reset_index()
    .sort_values(["dataset", "config_label"])
)

display(classification_summary)


## Summary Table


In [ ]:
summary_export_path = save_table(classification_summary, "classification", "classification_summary")
display(Markdown(f"Saved summary table to `{summary_export_path}`."))
classification_summary


## Main Figures


In [ ]:
datasets = sorted(classification_summary["dataset"].unique(), key=lambda x: DATASET_ORDER.get(x, 99))
fig, axes = plt.subplots(1, len(datasets), figsize=(20, 5), sharey=False)
axes = np.atleast_1d(axes)

for ax, dataset in zip(axes, datasets):
    ds = classification_summary[classification_summary["dataset"] == dataset].copy().reset_index(drop=True)
    x = np.arange(len(ds))
    ax.bar(x, ds["mean_accuracy"], color="#5e81ac")
    std_values = ds["std_accuracy"].fillna(0.0).to_numpy()
    ax.errorbar(x, ds["mean_accuracy"], yerr=std_values, fmt="none", ecolor="#2e3440", capsize=4)
    ax.set_title(pretty_dataset(dataset))
    ax.set_ylabel("Best accuracy (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(ds["config_label"], rotation=45, ha="right")

fig.suptitle("Dataset-wise Best Accuracy", fontsize=16)
fig.tight_layout()
main_path = save_figure(fig, "classification", "classification_best_accuracy")
display(Markdown(f"Saved main figure to `{main_path}`."))
plt.show()


## Secondary Figures


In [ ]:
datasets = sorted(classification["dataset"].unique(), key=lambda x: DATASET_ORDER.get(x, 99))
fig, axes = plt.subplots(len(datasets), 1, figsize=(14, 12), sharex=False)
axes = np.atleast_1d(axes)

for ax, dataset in zip(axes, datasets):
    ds = classification[classification["dataset"] == dataset].copy()
    for config_label, group in ds.groupby("config_label"):
        mean_curve, std_curve = mean_and_std(group["test_curve"])
        epochs = np.arange(1, len(mean_curve) + 1)
        ax.plot(epochs, mean_curve, linewidth=2.0, label=config_label)
        ax.fill_between(epochs, mean_curve - std_curve, mean_curve + std_curve, alpha=0.15)

    ax.set_title(pretty_dataset(dataset))
    ax.set_ylabel("Test metric")
    ax.legend(fontsize=8, ncol=2)

axes[-1].set_xlabel("Epoch")
fig.suptitle("Representative Mean Test Curves", fontsize=16)
fig.tight_layout()
curve_path = save_figure(fig, "classification", "classification_mean_test_curves")
display(Markdown(f"Saved test-curve figure to `{curve_path}`."))
plt.show()


In [ ]:
datasets = sorted(classification["dataset"].unique(), key=lambda x: DATASET_ORDER.get(x, 99))
fig, axes = plt.subplots(1, len(datasets), figsize=(20, 5), sharey=False)
axes = np.atleast_1d(axes)

for ax, dataset in zip(axes, datasets):
    ds = classification[classification["dataset"] == dataset].copy()
    labels = sorted(ds["config_label"].unique())
    positions = np.arange(len(labels))
    for pos, label in enumerate(labels):
        values = ds.loc[ds["config_label"] == label, "metric_percent"].to_numpy()
        ax.scatter(np.full_like(values, pos, dtype=float), values, s=45, alpha=0.8)

    ax.set_title(pretty_dataset(dataset))
    ax.set_ylabel("Best accuracy (%)")
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=45, ha="right")

fig.suptitle("Seed-Level Dispersion", fontsize=16)
fig.tight_layout()
dispersion_path = save_figure(fig, "classification", "classification_seed_dispersion")
display(Markdown(f"Saved dispersion figure to `{dispersion_path}`."))
plt.show()


## Export


In [ ]:
export_frame = classification.drop(columns=["train_curve", "test_curve"]).copy()
export_path = save_table(export_frame, "classification", "classification_runs")
display(Markdown(f"Saved normalized run table to `{export_path}`."))


## Notes

- PKS classification CSV는 single baseline 2개와 SSML pair 3개 구성을 dataset마다 제공합니다.
- pair 결과는 각 model의 관점으로 행을 분리해 `model | peer=...` 형식으로 비교합니다.
- metric은 higher-is-better accuracy이므로 summary chart는 백분율로 표시합니다.
